# EDA — Vector detections (VectorNet)

Initial load of the local VectorNet occurrence table for exploratory analysis.

**Prerequisite:** run [`01_extract_vectors.ipynb`](01_extract_vectors.ipynb) first so `data/raw/vector/` contains the latest DwC-A extract.

This notebook starts with reading the dataset via the shared `load_vector_detections` helper. Full cleaning, plots, and feature work come in later cells.

In [28]:
from __future__ import annotations

import json

from vector_guard.data import (
    default_vector_raw_dir,
    find_repo_root,
    load_vector_detections,
)

REPO_ROOT = find_repo_root()
RAW_DIR = default_vector_raw_dir(REPO_ROOT)
MANIFEST_PATH = RAW_DIR / "manifest.json"

df = load_vector_detections()
df.shape

(277353, 26)

## Source manifest

Metadata written by the extract notebook (fetch time, citation, checksum).

In [29]:
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print(f"fetched_at: {manifest.get('fetched_at')}")
    print(f"source_url: {manifest.get('source_url')}")
    print(f"sha256:     {manifest.get('sha256')}")
    print(f"citation:   {manifest.get('citation')}")
else:
    print(f"No manifest at {MANIFEST_PATH}")
    manifest = None
manifest

fetched_at: 2026-11-10T14:36:47.471240+00:00
source_url: https://ipt.gbif.org/archive.do?r=vndatabase
sha256:     5cff38259325a9b8d0a0b4226b42e763859780d4ec1d6d0f5a60958d2f2c1c79
citation:   VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority. VectorNet. Training Organization. Occurrence dataset (CC-BY 4.0). https://ipt.gbif.org/resource?r=vndatabase


{'source_url': 'https://ipt.gbif.org/archive.do?r=vndatabase',
 'resource': 'vndatabase',
 'gbif_dataset_uuid': 'e497586a-1bdf-4f69-90eb-645d615762c8',
 'fetched_at': '2026-11-10T14:36:47.471240+00:00',
 'zip_name': 'vectornet_dwca.zip',
 'zip_bytes': 30443144,
 'sha256': '5cff38259325a9b8d0a0b4226b42e763859780d4ec1d6d0f5a60958d2f2c1c79',
 'citation': 'VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority. VectorNet. Training Organization. Occurrence dataset (CC-BY 4.0). https://ipt.gbif.org/resource?r=vndatabase',
 'extracted_files': ['.gitkeep', 'eml.xml', 'meta.xml', 'occurrence.txt']}

## First look

In [30]:
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}):")
print(df.dtypes)

Rows: 277,353
Columns (26):
id                        str
bibliographicCitation     str
basisOfRecord             str
dynamicProperties         str
occurrenceID              str
individualCount           str
sex                       str
lifeStage                 str
degreeOfEstablishment     str
associatedTaxa            str
eventID                   str
eventDate                 str
samplingProtocol          str
sampleSizeValue           str
sampleSizeUnit            str
eventRemarks              str
locationID                str
higherGeographyID         str
higherGeography           str
countryCode               str
locationRemarks           str
decimalLatitude           str
decimalLongitude          str
verbatimIdentification    str
identificationRemarks     str
scientificName            str
dtype: object


In [31]:
df.head().T

,0,1,2,3,4
id,7E15E2D4-3F33-41A1-81DA-000025D68FE7,4B85477C-5310-4E4E-8AA7-00004B42A332,015BFCF3-1892-4F3A-AE1A-000058416ABE,549F3330-65E8-44CF-8882-00006605364E,75627364-A5C8-4143-B906-0000B500263B
bibliographicCitation,NaN,NaN,NaN,NaN,NaN
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
dynamicProperties,NaN,NaN,NaN,NaN,NaN
occurrenceID,7E15E2D4-3F33-41A1-81DA-000025D68FE7,4B85477C-5310-4E4E-8AA7-00004B42A332,015BFCF3-1892-4F3A-AE1A-000058416ABE,549F3330-65E8-44CF-8882-00006605364E,75627364-A5C8-4143-B906-0000B500263B
individualCount,0,25,0,0,0
sex,female,NaN,female,female,NaN
lifeStage,adult,egg,adult,adult,egg
degreeOfEstablishment,Native - Present,Invasive - Established,Native - Obs. Absent,Native - Present,Invasive - Absent
associatedTaxa,host: Not applicable,NaN,host: Not applicable,host: Not applicable,NaN


In [32]:
# Null summary with % of nulls over total rows
import pandas as pd
def calculate_null_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.isnull()
        .sum()
        .to_frame("null_count")
        .assign(
            null_pct=lambda x: (x["null_count"] / len(df) * 100).round(2),
            nunique=df.nunique(),
            dtype=df.dtypes,
            sample_values=df.apply(
                lambda s: s.dropna().unique()[:5].tolist()
            ),
        )
        .sort_values("null_count", ascending=False)
    )
calculate_null_summary(df)

,null_count,null_pct,nunique,dtype,sample_values
dynamicProperties,276827,99.81,12,str,"[{""pathogen"": ""Borrelia burgdorferi""}, {""patho..."
bibliographicCitation,232116,83.69,333,str,"[Balenghien T, DelÈcolle JC, Setier-Rio ML, Ma..."
associatedTaxa,158953,57.31,74,str,"[host: Not applicable, host: Bos taurus, host:..."
locationRemarks,154447,55.69,2,str,"[FALSE, Sheltered environment]"
sex,139012,50.12,4,str,"[female, male, other, male | female | other]"
eventRemarks,104267,37.59,4500,str,"[ABUN, DATASET DOI:10.15470/VS3677, ABUN_ANNUA..."
sampleSizeValue,81621,29.43,278,str,"[1, 7, 14, 2, 32]"
sampleSizeUnit,52349,18.87,15,str,"[traps * nights, traps * 24h, m3 * hour, dips,..."
samplingProtocol,51665,18.63,39,str,[Onderstepoort Veterinary Institute (OVI) blac...
degreeOfEstablishment,45276,16.32,7,str,"[Native - Present, Invasive - Established, Nat..."


# Fixing the countryCode nulls

In [33]:
# use the first two letters of the higherGeographyID to set the nulls in countryCode
df["countryCode"] = df["countryCode"].fillna(
    df["higherGeographyID"].astype("string").str[:2].str.upper()
)
df["countryCode"].isnull().sum()

np.int64(0)

In [34]:
#cheack that the countryCode is the same as the first two letters of the higherGeographyID
def country_code_mismatches(df: pd.DataFrame) -> pd.DataFrame:
    prefix = df["higherGeographyID"].astype(str).str[:2].str.upper()
    return df[df["countryCode"].ne(prefix)]

mismatches = country_code_mismatches(df)
mismatches

,id,bibliographicCitation,basisOfRecord,dynamicProperties,occurrenceID,individualCount,sex,lifeStage,degreeOfEstablishment,associatedTaxa,...,locationID,higherGeographyID,higherGeography,countryCode,locationRemarks,decimalLatitude,decimalLongitude,verbatimIdentification,identificationRemarks,scientificName


# split eventDate in start / end interval

In [40]:
# split eventDate in start / end interval date columns
parts = df["eventDate"].str.split("/", n=1, expand=True)
df["eventDate_start"] = pd.to_datetime(parts[0], errors="coerce")
df["eventDate_end"] = pd.to_datetime(parts[1], errors="coerce")

# single date (no "/") → treat as start = end
single = df["eventDate"].notna() & ~df["eventDate"].str.contains("/", na=False)
df.loc[single, "eventDate_end"] = df.loc[single, "eventDate_start"]

In [46]:
# inverted intervals (you already have at least one: 2015-09-07/2014-04-08)
bad = df[df["eventDate_end"] < df["eventDate_start"]]
#print(bad.sum())
bad.head().T


,4,8,23,131,140
id,75627364-A5C8-4143-B906-0000B500263B,32A455BA-92CD-4478-939B-00015830CE5F,F77BC260-A5AC-4359-87F2-000463E5782F,92D86BCF-E2AB-4BCB-A438-00208EABB000,7B87831E-BC84-4B50-BC2B-0022A3BCB781
bibliographicCitation,NaN,NaN,NaN,NaN,NaN
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
dynamicProperties,NaN,NaN,NaN,NaN,NaN
occurrenceID,75627364-A5C8-4143-B906-0000B500263B,32A455BA-92CD-4478-939B-00015830CE5F,F77BC260-A5AC-4359-87F2-000463E5782F,92D86BCF-E2AB-4BCB-A438-00208EABB000,7B87831E-BC84-4B50-BC2B-0022A3BCB781
individualCount,0,0,0,0,0
sex,NaN,female,male,female,NaN
lifeStage,egg,adult,adult,adult,egg
degreeOfEstablishment,Invasive - Absent,Native - Obs. Absent,Native - Present,Native - Obs. Absent,Invasive - Absent
associatedTaxa,NaN,host: Not applicable,host: Not applicable,host: Not applicable,NaN


In [ ]:
# parse failures / remaining odd formats
failed = df["eventDate"].notna() & df["eventDate_start"].isna()

In [36]:
null_summary = calculate_null_summary(df)
null_summary[null_summary["null_count"] < 0.1 * len(df)]

,null_count,null_pct,nunique,dtype,sample_values
lifeStage,27503,9.92,4,str,"[adult, egg, larva, nymph/pupa]"
eventDate,7933,2.86,14467,str,"[2024-04-07/2024-05-07, 2020-10-20/2020-10-27,..."
identificationRemarks,3308,1.19,8,str,"[morphological, PCR_DNA_Barcoding, PCR, MALDI-..."
higherGeography,1410,0.51,1773,str,"[Poland | Suwalski, Italy | Roma, Poland | Mia..."
basisOfRecord,0,0.00,1,str,[humanObservation]
id,0,0.00,277353,str,"[7E15E2D4-3F33-41A1-81DA-000025D68FE7, 4B85477..."
occurrenceID,0,0.00,277353,str,"[7E15E2D4-3F33-41A1-81DA-000025D68FE7, 4B85477..."
higherGeographyID,0,0.00,1819,str,"[PL843, ITI43, PL415, FRF34, CH011]"
locationID,0,0.00,25405,str,"[54.07396 : 23.37122, 42.07000 : 12.67139, 52...."
eventID,0,0.00,76134,str,"[049698b7-54b9-4004-8632-0a4a96efc4e1, f121252..."


In [ ]:
def print_col_info(col):
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print(df[col].value_counts().head())
    print()


print_col_info("eventDate")

eventDate
<StringArray>
['2024-04-07/2024-05-07', '2020-10-20/2020-10-27', '2024-05-05/2024-06-05',
 '2010-05-03/2010-06-03', '2015-09-07/2014-04-08']
Length: 5, dtype: str
14467
eventDate
2018-06-01/2018-09-30    10580
2020-01-01/2025-07-08     6277
2007-01-07/2008-01-07     2160
2020-01-01/2023-08-31     1879
2009-01-01/2009-12-31     1103
Name: count, dtype: int64

